In [1]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent

sys.path.insert(0, str(ROOT))

from scripts.evaluation.run_correlations import run_correlations
from scripts.evaluation.run_regression import vif_table, run_regression
from scripts.evaluation.run_loocv import loocv_summary, loocv_metrics
from scripts.evaluation.sample_filter import apply_two_part_rule, sensitivity_grid

### *Constants*

In [9]:
TRANSLUCENT = [
    "complexity_avg_raw", 
    "complexity_div", 
    "complexity_disp",  
    "complexity_diss_aa_seq",
]

BASELINE = [
    # Complexity.perform_measurements
    "magnitude",
    "support",
    "variety",
    "level_of_detail",
    "time_granularity",
    "structure",
    "affinity",
    "trace_length_min",
    "trace_length_avg",
    "trace_length_max",
    "distinct_traces",
    "deviation_from_random",
    "lempel_ziv_complexity",
    "pentland_task_complexity",
    "pentland_process_complexity",
    # Complexity.graph_complexity
    "variant_entropy",
    "variant_entropy_norm",
    # Complexity.log_complexity
    "sequence_entropy",
    "sequence_entropy_norm",
    "sequence_entropy_linear",
    "sequence_entropy_linear_norm",
    "sequence_entropy_exp",
    "sequence_entropy_exp_norm",
]

QUALITY = [
    "f_score_imtf", 
    "f_score_imts", 
    "f_score_imto", 
    "f_score_imftf", 
    "f_score_imfts", 
    "f_score_imfto",
]

CONTROL = [
    "num_activities"
]

# two-part rule for the analysis sample (see scripts/evaluation/sample_filter.py)
MIN_CASES = 30        # cases in the translucent log (stability of the metrics)
MIN_RETENTION = 0.10  # share of the cases that survived translucent generation (representativeness)

### *Group 1 (real_life and synthetic)*

In [3]:
from functools import reduce

import pandas as pd

FILES = ["complexity", "quality", "baseline_complexity", "activity_number", "retention"]


def load_dataset(folder: str) -> pd.DataFrame:
    """Read the result files of one dataset and inner-join them on the log name."""
    path = ROOT / "results" / "evaluation" / folder
    dfs = [pd.read_csv(path / f"{f}.csv") for f in FILES]
    df = reduce(lambda left, right: left.merge(right, on="name", how="inner", validate="1:1"), dfs)
    assert len(df) == len(dfs[0]), f"{folder}: inner join dropped logs"
    df = df.drop(columns=df.filter(regex=r"_(std|z)$").columns)
    return df.assign(dataset=folder)


# real-life and synthetic data (raw values, never modified)
rl_df = load_dataset("real_life")
sy_df = load_dataset("synthetic")
group1_raw_df = pd.concat([rl_df, sy_df], ignore_index=True)

print(group1_raw_df.shape)
print("\n")
print(group1_raw_df.columns)

(39, 58)


Index(['name', 'complexity_avg_raw', 'complexity_avg_norm', 'complexity_div',
       'complexity_disp', 'complexity_diss_aa_seq', 'complexity_diss_aa_mset',
       'complexity_diss_ac_seq', 'complexity_diss_ac_mset', 'precision_imts',
       'fitness_imts', 'f_score_imts', 'precision_imto', 'fitness_imto',
       'f_score_imto', 'precision_imtf', 'fitness_imtf', 'f_score_imtf',
       'precision_imfts', 'fitness_imfts', 'f_score_imfts', 'precision_imfto',
       'fitness_imfto', 'f_score_imfto', 'precision_imftf', 'fitness_imftf',
       'f_score_imftf', 'magnitude', 'support', 'variety', 'level_of_detail',
       'time_granularity', 'structure', 'affinity', 'trace_length_min',
       'trace_length_avg', 'trace_length_max', 'distinct_traces',
       'deviation_from_random', 'lempel_ziv_complexity',
       'pentland_task_complexity', 'pentland_process_complexity',
       'variant_entropy', 'variant_entropy_norm', 'sequence_entropy',
       'sequence_entropy_norm', 'sequence_e

### *Group 2 (simulated)*

In [4]:
# simulated data (raw values, never modified)
group2_raw_df = load_dataset("simulated")

print(group2_raw_df.shape)
print("\n")
print(group2_raw_df.columns)

(30, 58)


Index(['name', 'complexity_avg_raw', 'complexity_avg_norm', 'complexity_div',
       'complexity_disp', 'complexity_diss_aa_seq', 'complexity_diss_aa_mset',
       'complexity_diss_ac_seq', 'complexity_diss_ac_mset', 'precision_imts',
       'fitness_imts', 'f_score_imts', 'precision_imto', 'fitness_imto',
       'f_score_imto', 'precision_imtf', 'fitness_imtf', 'f_score_imtf',
       'precision_imfts', 'fitness_imfts', 'f_score_imfts', 'precision_imfto',
       'fitness_imfto', 'f_score_imfto', 'precision_imftf', 'fitness_imftf',
       'f_score_imftf', 'magnitude', 'support', 'variety', 'level_of_detail',
       'time_granularity', 'structure', 'affinity', 'trace_length_min',
       'trace_length_avg', 'trace_length_max', 'distinct_traces',
       'deviation_from_random', 'lempel_ziv_complexity',
       'pentland_task_complexity', 'pentland_process_complexity',
       'variant_entropy', 'variant_entropy_norm', 'sequence_entropy',
       'sequence_entropy_norm', 'sequence_e

### *Combine, Sample, and Standardize (two-part rule)*
Logs are kept when their translucent log has at least `MIN_CASES` cases and at least `MIN_RETENTION` of the cases it was made from survived translucent generation. All complexity measures (translucent and baseline) are standardized over the kept logs; quality measures and control variables stay raw.

In [5]:
def standardize(df: pd.DataFrame) -> pd.DataFrame:
    """Return a copy of df with all complexity measures (translucent and baseline) z-scored
    over the logs in df.

    Computed from the raw values, over the sample the analysis is run on (group 1, group 2
    or all logs), so that datasets are not standardized separately. ddof=0 as StandardScaler.
    Quality measures and control variables stay raw."""
    df = df.copy()
    measures = [*TRANSLUCENT, *BASELINE]
    x = df[measures]
    df[measures] = (x - x.mean()) / x.std(ddof=0)
    return df

# Combine
all_raw_df = pd.concat([group1_raw_df, group2_raw_df], ignore_index=True)

# Sample
# combined
sample_raw_df, excluded_df = apply_two_part_rule(all_raw_df, MIN_CASES, MIN_RETENTION)
sample_df = standardize(sample_raw_df)
# group1
sample_g1_raw_df, _ = apply_two_part_rule(group1_raw_df, MIN_CASES, MIN_RETENTION)
sample_g1_df = standardize(sample_g1_raw_df)
# group2
sample_g2_raw_df, _ = apply_two_part_rule(group2_raw_df, MIN_CASES, MIN_RETENTION)
sample_g2_df = standardize(sample_g2_raw_df)

print(f"kept {len(sample_df)} of {len(all_raw_df)} logs:", sample_df["dataset"].value_counts().to_dict())
print(f"\nexcluded ({len(excluded_df)}):")
print(excluded_df[["dataset", "name", "support", "retention", "reason"]]
      .sort_values(["dataset", "retention"]).to_string(index=False))

kept 57 of 69 logs: {'simulated': 24, 'synthetic': 18, 'real_life': 15}

excluded (12):
  dataset                  name  support  retention                 reason
real_life                  RTFM       14   0.000186 < 30 cases; < 10% kept
real_life     BPIC 13 Incidents        2   0.000530 < 30 cases; < 10% kept
real_life                Sepsis        2   0.003810 < 30 cases; < 10% kept
real_life             BPIC 12 O       14   0.005582 < 30 cases; < 10% kept
real_life BPIC 13 Open Problems       32   0.078049             < 10% kept
real_life Brandenburg Lawmaking       30   0.083565             < 10% kept
simulated      Simulated Log 32        1   0.001000 < 30 cases; < 10% kept
simulated       Simulated Log 6       44   0.044000             < 10% kept
simulated      Simulated Log 25       54   0.054000             < 10% kept
simulated      Simulated Log 10       59   0.059000             < 10% kept
simulated      Simulated Log 19       64   0.064000             < 10% kept
simulated   

### *Analysis*

In [10]:
T_COLUMNS = ["name", *TRANSLUCENT, *QUALITY, *CONTROL]
B_COLUMNS = ["name", *BASELINE, *QUALITY, *CONTROL]

#### Correlations - group 1

In [11]:
OUTPUT = ROOT / "results" / "evaluation" / "group1_corr"
outputs = [OUTPUT / "p_corr.csv", OUTPUT / "p_p_values.csv", OUTPUT / "k_corr.csv", OUTPUT / "k_p_values.csv"]

_, _, _, _ = run_correlations(sample_g1_df[T_COLUMNS], outputs, "translucent")
_, _, _, _ = run_correlations(sample_g1_df[B_COLUMNS], outputs, "baseline")

#### Correlations - group 2

In [12]:
OUTPUT = ROOT / "results" / "evaluation" / "group2_corr"
outputs = [OUTPUT / "p_corr.csv", OUTPUT / "p_p_values.csv", OUTPUT / "k_corr.csv", OUTPUT / "k_p_values.csv"]

_, _, _, _ = run_correlations(sample_g2_df[T_COLUMNS], outputs, "translucent")
_, _, _, _ = run_correlations(sample_g2_df[B_COLUMNS], outputs, "baseline")

#### Correlations - combined

In [13]:
OUTPUT = ROOT / "results" / "evaluation" / "corr"
outputs = [OUTPUT / "p_corr.csv", OUTPUT / "p_p_values.csv", OUTPUT / "k_corr.csv", OUTPUT / "k_p_values.csv"]

_, _, _, _ = run_correlations(sample_df[T_COLUMNS], outputs, "translucent")
_, _, _, _ = run_correlations(sample_df[B_COLUMNS], outputs, "baseline")

#### Regressions - combined

In [29]:
print(vif_table(sample_df, [*TRANSLUCENT, *CONTROL]))

                 variable       VIF
1      complexity_avg_raw  1.480104
2          complexity_div  1.225932
3         complexity_disp  2.133141
4  complexity_diss_aa_seq  1.623201
5          num_activities  2.022069


In [30]:
print(vif_table(sample_df, [*BASELINE, *CONTROL]))

                        variable           VIF
1                      magnitude    568.373626
2                        support    417.414486
3                        variety     26.054400
4                level_of_detail     36.173062
5               time_granularity      7.663870
6                      structure      8.683509
7                       affinity     12.820204
8               trace_length_min      7.173290
9               trace_length_avg     56.256665
10              trace_length_max     27.441171
11               distinct_traces     21.447227
12         deviation_from_random     11.215588
13         lempel_ziv_complexity     49.171636
14      pentland_task_complexity    137.714628
15   pentland_process_complexity      7.781253
16               variant_entropy    156.018663
17          variant_entropy_norm      7.410564
18              sequence_entropy  23436.307409
19         sequence_entropy_norm   1915.094191
20       sequence_entropy_linear   5093.553750
21  sequence_

In [31]:
REDUCED_BASELINE = [
    "trace_length_avg",
    "lempel_ziv_complexity",
    "affinity",
    "sequence_entropy_norm",
]

print(vif_table(sample_df, [*REDUCED_BASELINE, *CONTROL]))

                variable       VIF
1       trace_length_avg  1.658798
2  lempel_ziv_complexity  1.238330
3               affinity  2.966123
4  sequence_entropy_norm  2.994895
5         num_activities  1.070482


In [18]:
OUT_DIR = ROOT / "results" / "evaluation" / "reg"
run_regression(sample_df, QUALITY, TRANSLUCENT, OUT_DIR, "translucent")


--- f_score_imtf ---
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
const                      0.8296      0.017     48.404      0.000       0.795       0.864
complexity_avg_raw        -0.0566      0.018     -3.122      0.003      -0.093      -0.020
complexity_div            -0.0112      0.019     -0.590      0.558      -0.049       0.027
complexity_disp           -0.0437      0.021     -2.044      0.046      -0.087      -0.001
complexity_diss_aa_seq     0.0165      0.021      0.803      0.425      -0.025       0.058
n=57  R2=0.209  adj.R2=0.148  F p=0.0146

--- f_score_imts ---
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
const                      0.8995      0.013     67.218      0.000       0.873       0.926
compl

In [19]:
run_regression(sample_df, QUALITY, REDUCED_BASELINE, OUT_DIR, "baseline")


--- f_score_imtf ---
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
const                     0.8296      0.016     52.483      0.000       0.798       0.861
trace_length_avg         -0.0046      0.020     -0.228      0.820      -0.045       0.036
lempel_ziv_complexity    -0.0297      0.018     -1.691      0.097      -0.065       0.006
affinity                 -0.0523      0.027     -1.940      0.058      -0.106       0.002
sequence_entropy_norm    -0.0979      0.027     -3.579      0.001      -0.153      -0.043
n=57  R2=0.327  adj.R2=0.275  F p=0.0003

--- f_score_imts ---
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
const                     0.8995      0.015     62.007      0.000       0.870       0.929
trace_length_av

In [20]:
run_regression(sample_df, QUALITY, [*TRANSLUCENT, *REDUCED_BASELINE], OUT_DIR, "combined")


--- f_score_imtf ---
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
const                      0.8296      0.015     54.806      0.000       0.799       0.860
complexity_avg_raw        -0.0545      0.020     -2.727      0.009      -0.095      -0.014
complexity_div             0.0188      0.019      0.983      0.331      -0.020       0.057
complexity_disp           -0.0150      0.023     -0.655      0.515      -0.061       0.031
complexity_diss_aa_seq     0.0230      0.021      1.075      0.288      -0.020       0.066
trace_length_avg          -0.0426      0.024     -1.762      0.084      -0.091       0.006
lempel_ziv_complexity     -0.0251      0.018     -1.393      0.170      -0.061       0.011
affinity                  -0.0245      0.029     -0.831      0.410      -0.084       0.035
sequence_entropy_norm     -0.0609      0.033     -1.825      0.074  

#### *LOOCV - combined*

In [28]:
OUT_DIR = ROOT / "results" / "evaluation" / "reg"

print("--- LOOCV Summary - translucent complexity ---")
loocv_summary(sample_df, QUALITY, TRANSLUCENT, OUT_DIR, "translucent")
print("\n")
print("--- LOOCV Summary - baseline complexity ---")
loocv_summary(sample_df, QUALITY, REDUCED_BASELINE, OUT_DIR, "baseline")
print("\n")
print("--- LOOCV Summary - combined complexity ---")
loocv_summary(sample_df, QUALITY, [*TRANSLUCENT, *REDUCED_BASELINE], OUT_DIR, "combined")

--- LOOCV Summary - translucent complexity ---
              controls   n  R2_in  adjR2_in     Q2   RMSE
target                                                   
f_score_imtf            57  0.209     0.148 -0.288  0.158
f_score_imts            57  0.320     0.268  0.158  0.107
f_score_imto            57  0.186     0.123  0.004  0.198
f_score_imftf           57  0.126     0.058 -0.021  0.142
f_score_imfts           57  0.402     0.356 -0.036  0.107
f_score_imfto           57  0.130     0.063 -0.334  0.215


--- LOOCV Summary - baseline complexity ---
              controls   n  R2_in  adjR2_in     Q2   RMSE
target                                                   
f_score_imtf            57  0.327     0.275  0.140  0.129
f_score_imts            57  0.201     0.140 -0.031  0.119
f_score_imto            57  0.332     0.280  0.174  0.180
f_score_imftf           57  0.414     0.369  0.255  0.121
f_score_imfts           57  0.247     0.189  0.090  0.100
f_score_imfto           57  0.369    

#### *Controlled Group*

In [22]:
run_regression(sample_df, QUALITY, TRANSLUCENT, OUT_DIR, "translucent", controls=CONTROL)
run_regression(sample_df, QUALITY, REDUCED_BASELINE, OUT_DIR, "baseline", controls=CONTROL)
run_regression(sample_df, QUALITY, [*TRANSLUCENT, *REDUCED_BASELINE], OUT_DIR, "combined", controls=CONTROL)


--- f_score_imtf ---  (controls: num_activities)
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
const                      0.8897      0.051     17.552      0.000       0.788       0.991
complexity_avg_raw        -0.0438      0.021     -2.110      0.040      -0.085      -0.002
complexity_div            -0.0108      0.019     -0.573      0.569      -0.049       0.027
complexity_disp           -0.0600      0.025     -2.410      0.020      -0.110      -0.010
complexity_diss_aa_seq     0.0257      0.022      1.182      0.243      -0.018       0.069
num_activities            -0.0039      0.003     -1.259      0.214      -0.010       0.002
n=57  R2=0.232  adj.R2=0.157  F p=0.0164

--- f_score_imts ---  (controls: num_activities)
                             coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------

In [23]:
print("--- LOOCV Summary - translucent complexity - controlled ---")
loocv_summary(sample_df, QUALITY, TRANSLUCENT, OUT_DIR, "translucent", controls=CONTROL)
print("\n")
print("--- LOOCV Summary - baseline complexity -controlled ---")
loocv_summary(sample_df, QUALITY, REDUCED_BASELINE, OUT_DIR, "baseline", controls=CONTROL)
print("\n")
print("--- LOOCV Summary - combined complexity -controlled ---")
loocv_summary(sample_df, QUALITY, [*TRANSLUCENT, *REDUCED_BASELINE], OUT_DIR, "combined", controls=CONTROL)

--- LOOCV Summary - translucent complexity - controlled ---
                     controls   n  R2_in  adjR2_in     Q2   RMSE
target                                                          
f_score_imtf   num_activities  57  0.232     0.157 -0.296  0.158
f_score_imts   num_activities  57  0.322     0.256  0.133  0.109
f_score_imto   num_activities  57  0.210     0.132 -0.020  0.200
f_score_imftf  num_activities  57  0.222     0.146  0.013  0.139
f_score_imfts  num_activities  57  0.405     0.347 -0.057  0.108
f_score_imfto  num_activities  57  0.242     0.168 -0.286  0.211


--- LOOCV Summary - baseline complexity -controlled ---
                     controls   n  R2_in  adjR2_in     Q2   RMSE
target                                                          
f_score_imtf   num_activities  57  0.336     0.271  0.112  0.131
f_score_imts   num_activities  57  0.246     0.172 -0.041  0.119
f_score_imto   num_activities  57  0.368     0.306  0.150  0.183
f_score_imftf  num_activities  57  0.

#### *Influence diagnostics (Cook's distance)*
Cook's distance of every log in the baseline, translucent and combined models (with `CONTROL`), for each F-score. Logs above `COOK_THRESHOLD` (default 1) are flagged in any model. As a robustness check, the leave-one-out comparison is repeated without the flagged logs. This doesn't replace the main analysis on all logs.

In [24]:
from scripts.evaluation.run_regression import run_ols

COOK_THRESHOLD = 1.0
COOK_MODELS = {
    "baseline": REDUCED_BASELINE,
    "translucent": TRANSLUCENT,
    "mixed_all": [*REDUCED_BASELINE, *TRANSLUCENT],
}


def cooks_distances(df: pd.DataFrame, models: dict, targets: list, controls=None) -> pd.DataFrame:
    """Cook's distance of every log, for every model and target (one row per log, model and target)."""
    rows = []
    for model, predictors in models.items():
        for target in targets:
            fit, sub = run_ols(df, predictors, target, controls)
            for idx, d in zip(sub.index, fit.get_influence().cooks_distance[0]):
                rows.append({"model": model, "target": target, "name": df.loc[idx, "name"],
                             "dataset": df.loc[idx, "dataset"], "cooks_d": d})
    return pd.DataFrame(rows)


cooks_df = cooks_distances(sample_df, COOK_MODELS, QUALITY, CONTROL)

OUT_DIR = ROOT / "results" / "evaluation" / "reg"
OUT_DIR.mkdir(parents=True, exist_ok=True)
cooks_df.to_csv(OUT_DIR / "cooks_distance.csv", index=False)

# largest Cook's distance of each log per model (over the six F-scores)
max_d = cooks_df.pivot_table(index=["name", "dataset"], columns="model", values="cooks_d", aggfunc="max")
print(f"n = {len(sample_df)}, threshold = {COOK_THRESHOLD} (4/n = {4 / len(sample_df):.3f})")
print("\nlargest Cook's distance per log (top 10 by any model):")
print(max_d.loc[max_d.max(axis=1).sort_values(ascending=False).index[:10]].round(2).to_string())

flagged = cooks_df[cooks_df["cooks_d"] > COOK_THRESHOLD]
influential = sorted(flagged["name"].unique())
print(f"\nlogs with Cook's distance > {COOK_THRESHOLD}: {influential or 'none'}")
if len(flagged):
    print(flagged.sort_values("cooks_d", ascending=False).round(2).to_string(index=False))

n = 57, threshold = 1.0 (4/n = 0.070)

largest Cook's distance per log (top 10 by any model):
model                                         baseline  mixed_all  translucent
name                               dataset                                    
Synthetic Log 15                   synthetic      0.13       5.86         4.76
BPIC 12 A                          real_life      0.82       0.69         0.65
BPIC 20 Permit Log                 real_life      0.33       0.40         0.16
Simulated Log 16                   simulated      0.35       0.27         0.05
BPIC 20 International Declarations real_life      0.33       0.23         0.23
BPIC 12 W                          real_life      0.11       0.23         0.01
BPIC 17 Offer                      real_life      0.14       0.21         0.01
Simulated Log 9                    simulated      0.01       0.21         0.05
Hospital Billing                   real_life      0.20       0.20         0.03
Simulated Log 2                    si

In [27]:
import numpy as np

# robustness check: leave-one-out comparison with and without the flagged logs
robust_df = sample_df[~sample_df["name"].isin(influential)]


def mean_q2(df: pd.DataFrame, predictors: list, controls=None) -> list:
    q2 = [loocv_metrics(df, predictors, t, controls)["Q2"] for t in QUALITY]
    return [*q2, np.mean(q2)]


comparison = pd.DataFrame(
    {(sample, model): mean_q2(df, predictors, CONTROL)
     for sample, df in [(f"all ({len(sample_df)})", sample_df), (f"without flagged ({len(robust_df)})", robust_df)]
     for model, predictors in COOK_MODELS.items()},
    index=[*QUALITY, "mean"],
).T
comparison.to_csv(OUT_DIR / "cooks_robustness_q2.csv")
print(f"excluded: {influential or 'none'}\n")
print(comparison.round(2).to_string())

excluded: ['Synthetic Log 15']

                                  f_score_imtf  f_score_imts  f_score_imto  f_score_imftf  f_score_imfts  f_score_imfto  mean
all (57)             baseline             0.11         -0.04          0.15           0.30           0.10           0.31  0.16
                     translucent         -0.30          0.13         -0.02           0.01          -0.06          -0.29 -0.09
                     mixed_all            0.17          0.22         -0.17           0.26           0.16          -0.10  0.09
without flagged (56) baseline             0.10         -0.09          0.13           0.29           0.05           0.29  0.13
                     translucent         -0.01          0.03         -0.11          -0.03           0.19          -0.06  0.00
                     mixed_all            0.10          0.15          0.14           0.28           0.34           0.39  0.23
